# Clean CSV부터 날짜 단위 모델 검증

입력은 `ai_ready_time_features_clean.csv` 하나입니다. 원본 ZIP, 전류 CSV, FFT, 전처리 셀을 실행하지 않습니다.

날짜는 파일명의 `_YYYYMMDD_HHMMSS_`에서 추출합니다. 원본 메타데이터의 측정 시각과는 다를 수 있습니다. 기존 노트북의 최종 공통 날짜 9개와 클래스별 개수는 일치했지만, 전체 날짜표에서는 정상 1개가 11월 26/27일 사이에서 달랐습니다. 이 노트북은 공통 날짜 검증용이며 전체 날짜 메타데이터 복원을 보장하지 않습니다.

특징을 고르고 평가하는 데 같은 날짜를 사용한 탐색적 결과입니다. 실제 설비·부하 일반화 검증은 별도로 필요합니다.


In [10]:
from pathlib import Path
import re, json, sys, importlib.metadata
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

import os
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = Path(os.environ.get("MOTOR_FEATURE_DIR", str(PROJECT_ROOT / "data" / "processed")))
FEATURE_PATH = DATA / "ai_ready_time_features_clean.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "reproduction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean_df = pd.read_csv(FEATURE_PATH, encoding="utf-8-sig")
if clean_df["path"].duplicated().any():
    raise ValueError("중복 원본 경로가 있습니다.")
print("전체 샘플:", len(clean_df))
display(pd.crosstab(clean_df["equipment"], clean_df["label"]))


전체 샘플: 2993


label,베어링불량,정상,축정렬불량,회전체불평형
equipment,,,,
L-DSF-01,0,493,500,0
L-EF-04,0,500,0,500
L-SF-04,500,500,0,0


## 1. 파일명 날짜와 기존 검증 범위 확인


In [12]:
date_tokens = clean_df["path"].str.extract(r"_(\d{8})_\d{6}_", expand=False)
if date_tokens.isna().any():
    raise ValueError("날짜를 읽을 수 없는 파일명이 있습니다.")
clean_df["day"] = pd.to_datetime(date_tokens, format="%Y%m%d", errors="raise").dt.date
dsf_df = clean_df.loc[clean_df["equipment"].eq("L-DSF-01")].copy()
day_table = pd.crosstab(dsf_df["day"], dsf_df["label"])
display(day_table)
day_table.to_csv(OUTPUT_DIR / "filename_date_counts.csv")
overlap_days = day_table.index[(day_table["정상"] > 0) & (day_table["축정렬불량"] > 0)]
model_df = dsf_df.loc[dsf_df["day"].isin(overlap_days)].copy().reset_index(drop=True)
expected = {"2020-11-06":(1,19),"2020-11-07":(1,55),"2020-11-08":(3,56),
 "2020-11-09":(14,46),"2020-11-11":(3,45),"2020-11-12":(3,46),
 "2020-11-13":(4,50),"2020-11-15":(4,27),"2020-11-16":(5,25)}
actual = {str(d):(int(day_table.loc[d,"정상"]),int(day_table.loc[d,"축정렬불량"])) for d in overlap_days}
if actual != expected:
    raise ValueError(f"기존 검증 날짜/클래스 개수와 다릅니다: {actual}")
print("기존 검증 범위 일치:",len(overlap_days),"일 /",len(model_df),"개")
model_df["target"] = model_df["label"].eq("축정렬불량").astype(int)
model_df["mean_rms"] = model_df[["R_rms","S_rms","T_rms"]].mean(axis=1)
model_df["mean_p2p"] = model_df[["R_p2p","S_p2p","T_p2p"]].mean(axis=1)
p2p = model_df[["R_p2p","S_p2p","T_p2p"]]
model_df["p2p_imbalance_ratio"] = (p2p.max(axis=1)-p2p.min(axis=1))/p2p.mean(axis=1)
y = model_df["target"]
feature_sets = {
 "compact_4":["mean_rms","mean_p2p","rms_imbalance_ratio","p2p_imbalance_ratio"],
 "phase_7":["R_rms","S_rms","T_rms","R_p2p","S_p2p","T_p2p","rms_imbalance_ratio"],
 "phase_plus_imbalance_8":["R_rms","S_rms","T_rms","R_p2p","S_p2p","T_p2p","rms_imbalance_ratio","p2p_imbalance_ratio"]}
if not np.isfinite(model_df[feature_sets["phase_plus_imbalance_8"] + feature_sets["compact_4"]].to_numpy()).all():
    raise ValueError("모델 입력에 비유한 값이 있습니다.")


label,정상,축정렬불량
day,,
2020-11-03,0,16
2020-11-04,0,43
2020-11-05,0,14
2020-11-06,1,19
2020-11-07,1,55
2020-11-08,3,56
2020-11-09,14,46
2020-11-10,0,29
2020-11-11,3,45


기존 검증 범위 일치: 9 일 / 407 개


## 2. 모델과 검증 설정

XGBoost가 설치되지 않았다면 해당 실험만 건너뛰고 이를 출력합니다. 원본 수치 재현 비교를 위해 3-fold XGBoost 클래스 비율은 원본과 같이 전체 y에서 계산합니다. LODO는 훈련 fold에서 계산합니다.


In [29]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import balanced_accuracy_score, f1_score
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost 미설치: XGBoost/특징 집합 비교/SHAP는 미검증입니다.")
def logistic():
    return Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(
        class_weight="balanced",max_iter=1000,random_state=42))])
def xgb_model(target):
    return XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,
        subsample=0.8,colsample_bytree=0.8,
        scale_pos_weight=(target==0).sum()/(target==1).sum(),
        eval_metric="logloss",random_state=42,n_jobs=1)
splits = validation_days = [
    ["2020-11-06", "2020-11-07", "2020-11-08"],
    ["2020-11-11", "2020-11-12", "2020-11-13"],
    ["2020-11-09", "2020-11-15", "2020-11-16"],
]

day_strings = model_df["day"].astype(str)
splits = []

for fold, days in enumerate(validation_days, start=1):
    is_valid = day_strings.isin(days).to_numpy()

    train_idx = np.flatnonzero(~is_valid)
    valid_idx = np.flatnonzero(is_valid)
    splits.append((train_idx, valid_idx))

    print(
        f"Fold {fold}: train={len(train_idx)}, "
        f"valid={len(valid_idx)}, dates={days}"
    )


Fold 1: train=272, valid=135, dates=['2020-11-06', '2020-11-07', '2020-11-08']
Fold 2: train=256, valid=151, dates=['2020-11-11', '2020-11-12', '2020-11-13']
Fold 3: train=286, valid=121, dates=['2020-11-09', '2020-11-15', '2020-11-16']


## 3. 3-fold 모델 비교와 특징 집합 비교


In [31]:
models = {"Logistic":logistic(),"RandomForest":RandomForestClassifier(
    n_estimators=300,class_weight="balanced",random_state=42,n_jobs=1)}
if HAS_XGB: models["XGBoost"] = xgb_model(y)
try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(n_estimators=300,learning_rate=0.05,
        class_weight="balanced",random_state=42,verbose=-1,n_jobs=1)
except ImportError:
    print("LightGBM 미설치: 모델 비교에서 제외")
rows=[]
for name,model in models.items():
    scores=cross_validate(model,model_df[feature_sets["compact_4"]],y,cv=splits,
        scoring={"ba":"balanced_accuracy","f1":"f1_macro"},error_score="raise")
    rows.append({"model":name,"BA_mean":scores["test_ba"].mean(),
        "BA_std":scores["test_ba"].std(),"F1_macro":scores["test_f1"].mean()})
model_results=pd.DataFrame(rows)
display(model_results)
model_results.to_csv(OUTPUT_DIR/"model_comparison_3fold.csv",index=False)
if HAS_XGB:
    rows=[]
    for name,cols in feature_sets.items():
        scores=cross_validate(xgb_model(y),model_df[cols],y,cv=splits,
            scoring={"ba":"balanced_accuracy","f1":"f1_macro"},error_score="raise")
        rows.append({"feature_set":name,"BA_mean":scores["test_ba"].mean(),
            "BA_std":scores["test_ba"].std(),"F1_macro":scores["test_f1"].mean()})
    feature_results=pd.DataFrame(rows)
    display(feature_results)
    feature_results.to_csv(OUTPUT_DIR/"feature_set_comparison.csv",index=False)


,model,BA_mean,BA_std,F1_macro
0,Logistic,0.900435,0.091611,0.817827
1,RandomForest,0.823857,0.171039,0.798072
2,XGBoost,0.937803,0.031575,0.856605
3,LightGBM,0.912486,0.061518,0.928568


,feature_set,BA_mean,BA_std,F1_macro
0,compact_4,0.937803,0.031575,0.856605
1,phase_7,0.998818,0.001672,0.991470
2,phase_plus_imbalance_8,0.998818,0.001672,0.991470


## 4. 날짜 단위 OOF 검증

각 날짜를 한 번씩 평가합니다. 전체 OOF BA와 날짜 평균 BA를 구분합니다.


In [18]:
candidates={"R_rms_stump":(["R_rms"],DecisionTreeClassifier(max_depth=1,
    class_weight="balanced",random_state=42)),"Logistic_phase7":(feature_sets["phase_7"],logistic())}
if HAS_XGB:candidates["XGBoost_phase7"]=(feature_sets["phase_7"],None)
summary=[]; daily=[]; prediction_rows=[]
for name,(cols,estimator) in candidates.items():
    X=model_df[cols];oof=np.full(len(y),-1,dtype=int);fold_scores=[]
    for train,test in LeaveOneGroupOut().split(X,y,groups=model_df["day"]):
        fitted=xgb_model(y.iloc[train]) if name=="XGBoost_phase7" else clone(estimator)
        fitted.fit(X.iloc[train],y.iloc[train]);pred=fitted.predict(X.iloc[test]);oof[test]=pred
        ba=balanced_accuracy_score(y.iloc[test],pred);fold_scores.append(ba)
        daily.append({"model":name,"day":str(model_df.iloc[test[0]]["day"]),
            "n":len(test),"normal_n":int((y.iloc[test]==0).sum()),"fault_n":int((y.iloc[test]==1).sum()),
            "BA":ba,"F1_macro":f1_score(y.iloc[test],pred,average="macro")})
    assert (oof>=0).all()
    summary.append({"model":name,"OOF_BA":balanced_accuracy_score(y,oof),
        "OOF_F1_macro":f1_score(y,oof,average="macro"),"Day_BA_mean":np.mean(fold_scores),
        "Day_BA_std":np.std(fold_scores)})
    prediction_rows.append(pd.DataFrame({"model":name,"path":model_df["path"],
        "day":model_df["day"],"target":y,"prediction":oof}))
comparison_df=pd.DataFrame(summary);day_results_df=pd.DataFrame(daily)
display(comparison_df);display(day_results_df)
comparison_df.to_csv(OUTPUT_DIR/"lodo_model_comparison.csv",index=False)
day_results_df.to_csv(OUTPUT_DIR/"lodo_by_day.csv",index=False)
pd.concat(prediction_rows,ignore_index=True).to_csv(OUTPUT_DIR/"oof_predictions.csv",index=False)


,model,OOF_BA,OOF_F1_macro,Day_BA_mean,Day_BA_std
0,R_rms_stump,1.000000,1.000000,1.000000,0.000000
1,Logistic_phase7,0.993225,0.965725,0.990739,0.013713
2,XGBoost_phase7,0.998645,0.992828,0.998792,0.003416


,model,day,n,normal_n,fault_n,BA,F1_macro
0,R_rms_stump,2020-11-06,20,1,19,1.000000,1.000000
1,R_rms_stump,2020-11-07,56,1,55,1.000000,1.000000
2,R_rms_stump,2020-11-08,59,3,56,1.000000,1.000000
3,R_rms_stump,2020-11-09,60,14,46,1.000000,1.000000
4,R_rms_stump,2020-11-11,48,3,45,1.000000,1.000000
5,R_rms_stump,2020-11-12,49,3,46,1.000000,1.000000
6,R_rms_stump,2020-11-13,54,4,50,1.000000,1.000000
7,R_rms_stump,2020-11-15,31,4,27,1.000000,1.000000
8,R_rms_stump,2020-11-16,30,5,25,1.000000,1.000000
9,Logistic_phase7,2020-11-06,20,1,19,0.973684,0.819820


## 5. 전체 데이터 학습의 설명용 임계값과 SHAP

아래 학습은 OOF 성능 계산과 별개입니다. 원본 SHAP 모델 생성 코드는 누락돼 있었으므로 새로 명시한 설정이며 기존 SHAP 숫자 일치를 보장하지 않습니다.


In [20]:
stump=DecisionTreeClassifier(max_depth=1,class_weight="balanced",random_state=42)
stump.fit(model_df[["R_rms"]],y)
print(export_text(stump,feature_names=["R_rms"]))
if HAS_XGB:
    import xgboost as xgb
    X_shap=model_df[feature_sets["phase_7"]]
    fitted=xgb_model(y);fitted.fit(X_shap,y)
    contrib=fitted.get_booster().predict(xgb.DMatrix(X_shap),pred_contribs=True)
    importance=pd.Series(np.abs(contrib[:,:-1]).mean(axis=0),index=X_shap.columns).sort_values(ascending=False)
    display(importance)
    importance.rename("mean_abs_shap").to_csv(OUTPUT_DIR/"shap_importance.csv")
versions={"python":sys.version}
for name in ["numpy","pandas","scikit-learn","xgboost","lightgbm"]:
    try:versions[name]=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:versions[name]="not installed"
(OUTPUT_DIR/"environment.json").write_text(json.dumps(versions,indent=2),encoding="utf-8")
print("결과 위치:",OUTPUT_DIR.resolve())


|--- R_rms <= 2.30
|   |--- class: 0
|--- R_rms >  2.30
|   |--- class: 1



R_rms                  2.735217
S_rms                  0.605907
R_p2p                  0.180956
T_rms                  0.166223
rms_imbalance_ratio    0.014935
S_p2p                  0.000000
T_p2p                  0.000000
dtype: float32